# Semaine 3 — Modélisation : Régression Logistique

Ce notebook orchestre l'entraînement du modèle de Régression Logistique sur nos données pré-traitées, calcule les scores de leads (probabilités de conversion) et évalue les performances du modèle.

In [ ]:
import sys
import os

# Ajout du chemin racine du projet pour résoudre les imports de lead_scoring
sys.path.append(os.path.abspath(os.path.join("..", "..")))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import des fonctions du module de modélisation
from lead_scoring.src.model import entrainer_regression_logistique, predire_probabilites, sauvegarder_modele

## 1. Chargement des données pré-traitées

On recharge les données d'entraînement et de test normalisées et encodées lors de la Semaine 2.

In [ ]:
dossier_data = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "processed"))

X_train = pd.read_csv(os.path.join(dossier_data, "X_train.csv"))
X_test = pd.read_csv(os.path.join(dossier_data, "X_test.csv"))
y_train = pd.read_csv(os.path.join(dossier_data, "y_train.csv")).squeeze()
y_test = pd.read_csv(os.path.join(dossier_data, "y_test.csv")).squeeze()

print(f"Données d'entraînement : X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Données de test : X_test = {X_test.shape}, y_test = {y_test.shape}")

## 2. Entraînement du modèle

On entraîne la Régression Logistique sur nos données d'entraînement.

In [ ]:
modele_lr = entrainer_regression_logistique(X_train, y_train, random_state=42, max_iter=1000)

## 3. Prédiction des scores (Probabilités de conversion)

On calcule les scores (probabilités de souscription entre 0 et 1) pour les ensembles d'entraînement et de test.

In [ ]:
scores_train = predire_probabilites(modele_lr, X_train)
scores_test = predire_probabilites(modele_lr, X_test)

print(f"Scores d'entraînement (5 premiers) : {scores_train[:5].round(4)}")
print(f"Scores de test (5 premiers) : {scores_test[:5].round(4)}")

## 4. Visualisation de la distribution des scores

L'histogramme des scores permet de comprendre comment le modèle répartit les probabilités de conversion chez les prospects.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(scores_test, bins=20, kde=True, color="royalblue")
plt.title("Distribution des scores de leads (Test Set)")
plt.xlabel("Probabilité de souscription (Score)")
plt.ylabel("Nombre de prospects")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

## 5. Évaluation initiale des performances

On calcule l'AUC-ROC et on affiche le rapport de classification (Précision, Rappel, F1-score) en appliquant un seuil standard de 0.5.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# 1. Calcul de l'AUC-ROC
auc_train = roc_auc_score(y_train, scores_train)
auc_test = roc_auc_score(y_test, scores_test)
print(f"AUC-ROC (Entraînement) : {auc_train:.4f}")
print(f"AUC-ROC (Test)         : {auc_test:.4f}")

# 2. Rapport de classification à un seuil de 0.5
predictions_test = (scores_test >= 0.5).astype(int)
print("\nRapport de classification (Test Set) :")
print(classification_report(y_test, predictions_test))

## 6. Sauvegarde du modèle entraîné

On sauvegarde le modèle entraîné sous format `.joblib` dans le dossier racine `models/`.

In [ ]:
chemin_sauvegarde = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "models", "regression_logistique.joblib"))
sauvegarder_modele(modele_lr, chemin_sauvegarde)